# Financial Data Extraction and Visualization Dashboard

This notebook extracts historical stock price and quarterly revenue data for **Tesla (TSLA)**, **Amazon (AMZN)**, **AMD (AMD)**, and **GameStop (GME)** using the `yfinance` library and web scraping techniques (BeautifulSoup). We then build an interactive Plotly dashboard to visualize and analyze the trends.

In [1]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

/Users/mac/Desktop/StockAnalysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def make_graph(stock_data, revenue_data, stock_name):
    """
    Creates an interactive dashboard with two subplots:
    1. Historical Share Price (top)
    2. Historical Revenue (bottom)
    Using Plotly on shared x-axes.
    """
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        subplot_titles=("Historical Share Price", "Historical Revenue"), 
        vertical_spacing=0.15
    )
    
    # 1. Share Price (Row 1)
    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(stock_data.Date), 
            y=stock_data.Close.astype(float), 
            name="Share Price",
            line=dict(color="#1f77b4", width=2)
        ),
        row=1, col=1
    )
    
    # 2. Revenue (Row 2)
    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(revenue_data.Date), 
            y=revenue_data.Revenue.astype(float), 
            name="Quarterly Revenue (Millions)",
            line=dict(color="#ff7f0e", width=2)
        ),
        row=2, col=1
    )
    
    # Update layout
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($ Millions)", row=2, col=1)
    
    fig.update_layout(
        title=dict(
            text=f"{stock_name} Stock and Revenue Dashboard",
            x=0.5,
            xanchor='center',
            font=dict(size=18, color="white")
        ),
        showlegend=True,
        template="plotly_dark",
        height=600,
        margin=dict(t=80, b=50, l=60, r=40),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Add Range Slider to Row 1 x-axis
    fig.update_layout(xaxis=dict(rangeslider=dict(visible=True), type="date"))
    
    fig.show()

## Section 1: Tesla Stock and Revenue Data

In [3]:
# Create TSLA ticker object
tesla_ticker = yf.Ticker("TSLA")

# Extract historical stock data
tesla_data = tesla_ticker.history(period="max")

# Reset index to make Date a column
tesla_data.reset_index(inplace=True)

# Format Date to string/datetime format without timezone
tesla_data['Date'] = pd.to_datetime(tesla_data['Date']).dt.tz_localize(None)

# Display the first 5 rows
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [4]:
# Fetch HTML content
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text

# Parse HTML
soup = BeautifulSoup(html_data, "html.parser")

# Find the table containing "Tesla Quarterly Revenue" (Table 1)
tables = soup.find_all("table")
quarterly_table = tables[1]

# Extract rows
rows = []
for row in quarterly_table.find_all("tr")[1:]: # skip header
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        rows.append({"Date": date, "Revenue": revenue})

tesla_revenue = pd.DataFrame(rows)

# Clean revenue data: remove commas, dollar signs
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r"[,$]", "", regex=True)

# Drop any rows with null or empty values
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

# Convert to numeric
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"])

# Display the last 5 rows
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


In [5]:
# Plot Tesla Dashboard
make_graph(tesla_data, tesla_revenue, "Tesla")

## Section 2: GameStop Stock and Revenue Data

In [6]:
# Create GME ticker object
gme_ticker = yf.Ticker("GME")

# Extract historical stock data
gme_data = gme_ticker.history(period="max")

# Reset index to make Date a column
gme_data.reset_index(inplace=True)

# Format Date to string/datetime format without timezone
gme_data['Date'] = pd.to_datetime(gme_data['Date']).dt.tz_localize(None)

# Display the first 5 rows
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [7]:
# Fetch HTML content
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data = requests.get(url).text

# Parse HTML
soup = BeautifulSoup(html_data, "html.parser")

# Find the table containing "GameStop Quarterly Revenue" (Table 1)
tables = soup.find_all("table")
quarterly_table = tables[1]

# Extract rows
rows = []
for row in quarterly_table.find_all("tr")[1:]: # skip header
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        rows.append({"Date": date, "Revenue": revenue})

gme_revenue = pd.DataFrame(rows)

# Clean revenue data: remove commas, dollar signs
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r"[,$]", "", regex=True)

# Drop any rows with null or empty values
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]

# Convert to numeric
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"])

# Display the last 5 rows
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


In [8]:
# Plot GameStop Dashboard
make_graph(gme_data, gme_revenue, "GameStop")

## Section 3: Amazon Stock and Revenue Data

In [9]:
# Create AMZN ticker object
amazon_ticker = yf.Ticker("AMZN")

# Extract historical stock data
amazon_data = amazon_ticker.history(period="max")

# Reset index to make Date a column
amazon_data.reset_index(inplace=True)

# Format Date to string/datetime format without timezone
amazon_data['Date'] = pd.to_datetime(amazon_data['Date']).dt.tz_localize(None)

# Display the first 5 rows
amazon_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1997-05-15,0.121875,0.125000,0.096354,0.097917,1443120000,0.0,0.0
1,1997-05-16,0.098438,0.098958,0.085417,0.086458,294000000,0.0,0.0
2,1997-05-19,0.088021,0.088542,0.081250,0.085417,122136000,0.0,0.0
3,1997-05-20,0.086458,0.087500,0.081771,0.081771,109344000,0.0,0.0
4,1997-05-21,0.081771,0.082292,0.068750,0.071354,377064000,0.0,0.0


In [10]:
# Fetch HTML content from Macrotrends
url = "https://www.macrotrends.net/stocks/charts/AMZN/amazon/revenue"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
html_data = requests.get(url, headers=headers).text

# Parse HTML
soup = BeautifulSoup(html_data, "html.parser")

# Find the table containing "Amazon Quarterly Revenue" (Table 1)
tables = soup.find_all("table")
quarterly_table = tables[1]

# Extract rows
rows = []
for row in quarterly_table.find_all("tr")[1:]:
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        rows.append({"Date": date, "Revenue": revenue})

amazon_revenue = pd.DataFrame(rows)

# Clean revenue data: remove commas, dollar signs
amazon_revenue["Revenue"] = amazon_revenue["Revenue"].str.replace(r"[,$]", "", regex=True)

# Drop any rows with null or empty values
amazon_revenue.dropna(inplace=True)
amazon_revenue = amazon_revenue[amazon_revenue['Revenue'] != ""]

# Convert to numeric
amazon_revenue["Revenue"] = pd.to_numeric(amazon_revenue["Revenue"])

# Display the last 5 rows
amazon_revenue.tail()

,Date,Revenue
55,2012-06-30,12834
56,2012-03-31,13185
57,2011-12-31,17431
58,2011-09-30,10876
59,2011-06-30,9913


In [11]:
# Plot Amazon Dashboard
make_graph(amazon_data, amazon_revenue, "Amazon")

## Section 4: AMD Stock and Revenue Data

In [12]:
# Create AMD ticker object
amd_ticker = yf.Ticker("AMD")

# Extract historical stock data
amd_data = amd_ticker.history(period="max")

# Reset index to make Date a column
amd_data.reset_index(inplace=True)

# Format Date to string/datetime format without timezone
amd_data['Date'] = pd.to_datetime(amd_data['Date']).dt.tz_localize(None)

# Display the first 5 rows
amd_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,1980-03-17,3.125000,3.302083,3.125000,3.145833,219600,0.0,0.0
1,1980-03-18,3.125000,3.125000,2.937500,3.031250,727200,0.0,0.0
2,1980-03-19,3.031250,3.083333,3.020833,3.041667,295200,0.0,0.0
3,1980-03-20,3.041667,3.062500,3.010417,3.010417,159600,0.0,0.0
4,1980-03-21,3.010417,3.020833,2.906250,2.916667,130800,0.0,0.0


In [13]:
# Fetch HTML content from Macrotrends
url = "https://www.macrotrends.net/stocks/charts/AMD/advanced-micro-devices/revenue"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
html_data = requests.get(url, headers=headers).text

# Parse HTML
soup = BeautifulSoup(html_data, "html.parser")

# Find the table containing "AMD Quarterly Revenue" (Table 1)
tables = soup.find_all("table")
quarterly_table = tables[1]

# Extract rows
rows = []
for row in quarterly_table.find_all("tr")[1:]:
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip()
        rows.append({"Date": date, "Revenue": revenue})

amd_revenue = pd.DataFrame(rows)

# Clean revenue data: remove commas, dollar signs
amd_revenue["Revenue"] = amd_revenue["Revenue"].str.replace(r"[,$]", "", regex=True)

# Drop any rows with null or empty values
amd_revenue.dropna(inplace=True)
amd_revenue = amd_revenue[amd_revenue['Revenue'] != ""]

# Convert to numeric
amd_revenue["Revenue"] = pd.to_numeric(amd_revenue["Revenue"])

# Display the last 5 rows
amd_revenue.tail()

,Date,Revenue
55,2012-06-30,1413
56,2012-03-31,1585
57,2011-12-31,1691
58,2011-09-30,1690
59,2011-06-30,1574


In [14]:
# Plot AMD Dashboard
make_graph(amd_data, amd_revenue, "AMD")

# Summary of Findings

### Data Analysis Key Findings
* **Tesla (TSLA)** shows a strong long-term alignment between quarterly revenue growth and stock price appreciation. Revenue surged dramatically starting in 2020, aligned with an exponential rise in its share price.
* **Amazon (AMZN)** exhibits exceptionally consistent revenue growth with recurring seasonal peaks in Q4 (due to holiday shopping). Stock performance tracks the revenue upward trend closely over the past decade.
* **AMD (AMD)** displays cyclical semiconductor revenue patterns, showing periods of flat/dipping revenues followed by massive spikes corresponding to product release cycles (Ryzen/EPYC) and AI chip demand. The stock price behaves leading/coincident with these trends.
* **GameStop (GME)** shows flat to declining quarterly revenues over the last 5-10 years. However, its stock price experienced a massive, unprecedented spike in early 2021. This indicates the 2021 price action was driven by market dynamics (retail short squeeze) rather than company revenue fundamentals.

### Insights or Next Steps
* Quantify correlation: Perform a rolling Pearson correlation coefficient analysis between quarterly revenue growth and stock price movements to measure direct statistical coupling.
* Incorporate non-fundamental variables: Since GME showed that stock price can decouple completely from revenues, overlay social media sentiment metrics (like volume of mentions on Reddit or Twitter) onto the chart to analyze non-fundamental price movements.